In [2]:
import os, re
from typing import List, Dict
from docx import Document

from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader

# LOADING

In [5]:
dir_loader = DirectoryLoader(
    r"C:/Users/Lenovo/Desktop/Resume+DJ match maker/docc",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_documents = dir_loader.load()

print("Total docs loaded:", len(pdf_documents))
print(pdf_documents[0].page_content[:500])


100%|██████████| 1/1 [00:00<00:00,  2.37it/s]

Total docs loaded: 1
Devanshi Faldu
6351478229 • devanshifaldu@gmail.com • Banglore, Karnataka
EDUCATION
DAYANAND SAGAR UNIVERSITY | Masters of Science
Data Science | Expected May 2027 | Banglore, Karnataka
Course Works: Mathematics for Data Science, Python Programming, Introduction to Artificial Intelligence, Data Visualization,
Data Mining, Fundamentals of Data Science
MARWADI UNIVERSITY | Bachelors of Computer Application
July 2022 - March 2025 | Rajkot, Gujarat • Cum. GPA: 8.35/10
Course works: Web Designing, Co


# CHUNKING

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", "•", "-", " ", ""]
)

resume_chunks = splitter.split_documents(pdf_documents)

print("Total chunks:", len(resume_chunks))
print("First chunk:\n", resume_chunks[0].page_content)

Total chunks: 6
First chunk:
 Devanshi Faldu
6351478229 • devanshifaldu@gmail.com • Banglore, Karnataka
EDUCATION
DAYANAND SAGAR UNIVERSITY | Masters of Science
Data Science | Expected May 2027 | Banglore, Karnataka
Course Works: Mathematics for Data Science, Python Programming, Introduction to Artificial Intelligence, Data Visualization,
Data Mining, Fundamentals of Data Science
MARWADI UNIVERSITY | Bachelors of Computer Application
July 2022 - March 2025 | Rajkot, Gujarat • Cum. GPA: 8.35/10


In [ ]:
for i, d in enumerate(resume_chunks[:6]):  # show  all 6 chunks
    page = d.metadata.get("page", "NA")
    print(f"\n--- Chunk {i} | Page {page} ---\n")
    print(d.page_content[:900])


--- Chunk 0 | Page 0 ---

Devanshi Faldu
6351478229 • devanshifaldu@gmail.com • Banglore, Karnataka
EDUCATION
DAYANAND SAGAR UNIVERSITY | Masters of Science
Data Science | Expected May 2027 | Banglore, Karnataka
Course Works: Mathematics for Data Science, Python Programming, Introduction to Artificial Intelligence, Data Visualization,
Data Mining, Fundamentals of Data Science
MARWADI UNIVERSITY | Bachelors of Computer Application
July 2022 - March 2025 | Rajkot, Gujarat • Cum. GPA: 8.35/10

--- Chunk 1 | Page 0 ---

July 2022 - March 2025 | Rajkot, Gujarat • Cum. GPA: 8.35/10
Course works: Web Designing, Computer Networks, Data Structure, Algorithms, Database Management Systems, Operating
System, Android Development.
EXPERIENCE
AI INTERN Nexeo Security
September 2024 - March 2025 | Remote
•Developed and optimized AI-driven threat detection models that improved anomaly identification accuracy by over 25%,
enhancing the organization’s proactive cybersecurity measures.

--- Chunk 2 | Pag

# preparing job dscriptions for embedding

In [24]:
job_discription_text = """Role Overview
Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (quantization, pruning, distillation) for edge or cloud deployment.

Core Responsibilities
Productionalizing ML Models: Building, testing, and deploying ML models to production, focusing on robustness and efficiency.
Infrastructure Development: Building tools and infrastructure for distributed computing, model deployment, evaluation, and monitoring.
Algorithm Implementation: Implementing advanced AI techniques, including Large Language Models (LLMs), Computer Vision, and Neural Networks.
Performance Optimization: Optimizing models for low-latency inference on mobile devices (Android) and cloud environments.
Cross-functional Collaboration: Partnering with research teams (e.g., Google DeepMind) to apply new AI techniques to products like Search, YouTube, and Workspace.
Technical Advising: Acting as a subject matter expert, creating documentation, and conducting code reviews.

Key Areas of Specialization
Generative AI & LLMs: Developing RAG (Retrieval-Augmented Generation) systems, fine-tuning, and managing large-scale, multi-modal models.
On-Device AI: Building personalized, private AI experiences directly on mobile or desktop devices.
Conversational AI: Creating chatbot or voicebot applications using platforms like Dialogflow.
Trust & Safety: Building guardrails and classifiers to mitigate AI risks and misuse.

Minimum Qualifications
Education: Bachelor’s degree in Computer Science, or equivalent practical experience.
Experience: 3+ years of experience in software development and building ML solutions.
Programming: Proficiency in Python, C++, Go, or Java, with a strong understanding of data structures and algorithms.
ML Frameworks: Experience with TensorFlow, JAX, or PyTorch.

Preferred Qualifications
Advanced Degree: Master's or PhD in Computer Science, Image Processing, or Machine Learning.
System Design: Experience designing cloud enterprise solutions and distributed systems.
Infrastructure Expertise: Familiarity with data pipelines, MLOps, and data warehousing concepts (e.g., Apache Beam, Spark).
"""

# convert the text to a langchain document

In [25]:
from langchain_core.documents import Document

jd_doc = Document(
    page_content=job_discription_text,
    metadata={"source": "jd_google_ai_engineer"}
)

# chunking the job descriptions documents we made above

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

job_discriptions_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=80,
    separators=["\n\n", "\n", "•", "-", ": ", " ", ""]
)

job_discription_chunks = job_discriptions_splitter.split_documents([jd_doc])

print("Job Discription chunks:", len(job_discription_chunks))
print(job_discription_chunks[4].page_content)

Job Discription chunks: 6
Minimum Qualifications
Education: Bachelor’s degree in Computer Science, or equivalent practical experience.
Experience: 3+ years of experience in software development and building ML solutions.
Programming: Proficiency in Python, C++, Go, or Java, with a strong understanding of data structures and algorithms.
ML Frameworks: Experience with TensorFlow, JAX, or PyTorch.


In [34]:
for i, c in enumerate(job_discription_chunks):
    print(f"\n--- Job Discription Chunk {i} ---\n{c.page_content}")


--- Job Discription Chunk 0 ---
Role Overview
Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (quantization, pruning, distillation) for edge or cloud deployment.

--- Job Discription Chunk 1 ---
Core Responsibilities
Productionalizing ML Models: Building, testing, and deploying ML models to production, focusing on robustness and efficiency.
Infrastructure Development: Building tools and infrastructure for distributed computing, model deployment, evaluation, and monitoring.
Algorithm Implementation: Implementing advanced AI techniques, including Large Language Models (LLMs), Computer Vision, and Neural Networks.

--- Job Discription Chunk 2 ---
Performance Optimization: Optimizing models for low-latency inference on mobile devices (Android) and cloud enviro

In [35]:
print("resume_chunks:", len(resume_chunks))
print("jd_chunks:", len(job_discription_chunks))

print("\nSample resume chunk:\n", resume_chunks[0].page_content[:300])
print("\nSample JD chunk:\n", job_discription_chunks[0].page_content[:300])

resume_chunks: 6
jd_chunks: 6

Sample resume chunk:
 Devanshi Faldu
6351478229 • devanshifaldu@gmail.com • Banglore, Karnataka
EDUCATION
DAYANAND SAGAR UNIVERSITY | Masters of Science
Data Science | Expected May 2027 | Banglore, Karnataka
Course Works: Mathematics for Data Science, Python Programming, Introduction to Artificial Intelligence, Data Visu

Sample JD chunk:
 Role Overview
Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (quan


# building vetor db (chorma) for resume chunks

In [37]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


In [38]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=resume_chunks,
    embedding=embeddings,
    collection_name="resume_chunks"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("OpenAI Vector DB ready ✅")

100%|██████████| 1/1 [1:17:49<00:00, 4669.96s/it]


OpenAI Vector DB ready ✅


# retrieval matching between jd chunks and best resume chunks

In [40]:
matches = []

for i, jd in enumerate(job_discription_chunks):  
    hits = retriever.invoke(jd.page_content)    
    matches.append({
        "jd_chunk_id": i,
        "jd_text": jd.page_content.strip(),
        "resume_hits": hits
    })

print("Matched ✅", len(matches), "JD chunks")

Matched ✅ 6 JD chunks


In [41]:
for m in matches:
    print("\n" + "="*110)
    print(f"JD CHUNK {m['jd_chunk_id']}:\n{m['jd_text']}\n")
    print("-"*110)
    for j, h in enumerate(m["resume_hits"], start=1):
        page = h.metadata.get("page", "NA")
        print(f"\n[Resume Hit {j}] Page: {page}")
        print(h.page_content[:700])


JD CHUNK 0:
Role Overview
Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (quantization, pruning, distillation) for edge or cloud deployment.

--------------------------------------------------------------------------------------------------------------

[Resume Hit 1] Page: 0
July 2022 - March 2025 | Rajkot, Gujarat • Cum. GPA: 8.35/10
Course works: Web Designing, Computer Networks, Data Structure, Algorithms, Database Management Systems, Operating
System, Android Development.
EXPERIENCE
AI INTERN Nexeo Security
September 2024 - March 2025 | Remote
•Developed and optimized AI-driven threat detection models that improved anomaly identification accuracy by over 25%,
enhancing the organization’s proactive cybersecurity measures.

[Resume Hit 2] Page: 0
exper